In [1]:
from pyspark.sql import SparkSession

# Creating Spark Session
spark = (SparkSession.builder
    .appName("BigData_Lab02.1_Ubuntu") # Spark session name
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1")\
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "2g")
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
            "org.apache.kafka:kafka-clients:3.6.0,"
            "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.1") 
    .getOrCreate())

# Checking success
print(f"Successfully! Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/27 22:06:57 WARN Utils: Your hostname, Kiennguyen, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 22:06:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/kiennguyen/.ivy2.5.2/cache
The jars for the packages stored in: /home/kiennguyen/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
org.apache.spark#spark-streaming-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2aea98a7-5c79-44c3-845a-13fc244390ff;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central

Successfully! Spark version: 4.0.1


In [ ]:
spark.stop()

In [2]:
import kagglehub

# Download dataset from kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# reading data to spark frame
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

# Checking by show 1 row of each df
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row
+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row


In [3]:
from confluent_kafka.admin import AdminClient

admin_client = AdminClient({
    'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'
})

topics_to_delete = ["Lab1_ratings", "Lab1_movies", "Lab1_tags"]

fs = admin_client.delete_topics(topics_to_delete)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' deleted.")
    except Exception as e:
        print(f"Error deleting topic {topic}: {e}")

Topic 'Lab1_ratings' deleted.
Topic 'Lab1_movies' deleted.
Topic 'Lab1_tags' deleted.


In [4]:
from confluent_kafka.admin import AdminClient, NewTopic

# Connect to brokers
admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})

# Setting
new_topics = [
    NewTopic(topic="Lab1_ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="Lab1_movies", num_partitions=1, replication_factor=3),
    NewTopic(topic="Lab1_tags", num_partitions=1, replication_factor=3)
]

# Creating topics
fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' created.")
    except Exception as e:
        print(f"Error when creating topic {topic}: {e}")

Topic 'Lab1_ratings' created.
Topic 'Lab1_movies' created.
Topic 'Lab1_tags' created.


In [8]:
import time
from pyspark.sql.functions import struct, to_json

# Kafka bootstrap servers
brokers = "localhost:9092,localhost:9192,localhost:9292"

def push_all_topics_simultaneously(df_list_with_topics, batch_size=100, delay=2):
    """
    df_list_with_topics: Danh sách các tuple [(df, topic_name), ...]
    """
    # 1. Chuyển tất cả DataFrame thành danh sách JSON trước để xử lý nhanh trong vòng lặp
    queues = []
    for df, topic in df_list_with_topics:
        print(f"Đang chuẩn bị dữ liệu cho topic: {topic}...")
        json_data = df.selectExpr("to_json(struct(*)) AS value").collect()
        queues.append({
            "data": json_data,
            "topic": topic,
            "length": len(json_data)
        })

    # 2. Tìm độ dài lớn nhất để vòng lặp chạy đến khi hết dữ liệu của bảng dài nhất
    max_rows = max(q["length"] for q in queues)

    print("\n>>> Bắt đầu đẩy dữ liệu đồng bộ vào Kafka...")
    
    for i in range(0, max_rows, batch_size):
        # Trong mỗi lượt, duyệt qua từng topic để push 100 dòng
        for q in queues:
            start_idx = i
            end_idx = min(i + batch_size, q["length"])
            
            # Chỉ push nếu topic đó vẫn còn dữ liệu
            if start_idx < q["length"]:
                batch = q["data"][start_idx : end_idx]
                batch_df = spark.createDataFrame(batch)
                
                batch_df.write \
                    .format("kafka") \
                    .option("kafka.bootstrap.servers", brokers) \
                    .option("topic", q["topic"]) \
                    .save()
                
                print(f"[{q['topic']}] Đã đẩy từ dòng {start_idx} đến {end_idx}")

        print(f"--- Hoàn thành đợt {i//batch_size + 1}. Nghỉ {delay} giây ---\n")
        time.sleep(delay)

# Gọi hàm thực hiện
push_all_topics_simultaneously([
    (df_ratings, "Lab1_ratings"),
    (df_movies, "Lab1_movies"),
    (df_tags, "Lab1_tags")
], batch_size=100, delay=2)

Đang chuẩn bị dữ liệu cho topic: Lab1_ratings...
Đang chuẩn bị dữ liệu cho topic: Lab1_movies...
Đang chuẩn bị dữ liệu cho topic: Lab1_tags...

>>> Bắt đầu đẩy dữ liệu đồng bộ vào Kafka...
[Lab1_ratings] Đã đẩy từ dòng 0 đến 100


[Lab1_movies] Đã đẩy từ dòng 0 đến 100


[Lab1_tags] Đã đẩy từ dòng 0 đến 100
--- Hoàn thành đợt 1. Nghỉ 2 giây ---



[Lab1_ratings] Đã đẩy từ dòng 100 đến 200


[Lab1_movies] Đã đẩy từ dòng 100 đến 200


...

[Lab1_ratings] Đã đẩy từ dòng 700 đến 800


[Lab1_movies] Đã đẩy từ dòng 700 đến 800


ERROR:root:KeyboardInterrupt while sending command.               (0 + 12) / 12]
Traceback (most recent call last):
  File "/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
26/04/27 22:28:59 ERROR Executor: Exception in task 8.0 in stage 129.0 (TID 1347)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", lin

KeyboardInterrupt: 

l.kafka010.KafkaWriter$.$anonfun$write$1(KafkaWriter.scala:75)
	at org.apache.spark.sql.kafka010.KafkaWriter$.$anonfun$write$1$adapted(KafkaWriter.scala:72)
	at org.apache.spark.rdd.RDD.$anonfun$foreachPartition$2(RDD.scala:1047)
	at org.apache.spark.rdd.RDD.$anonfun$foreachPartition$2$adapted(RDD.scala:1047)
	at org.apache.spark.SparkContext.$anonfun$runJob$5(SparkContext.scala:2524)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:93)
	at org.apache.spark.TaskContext.runTaskWithListeners(TaskContext.scala:171)
	at org.apache.spark.scheduler.Task.run(Task.scala:147)
	at org.apache.spark.executor.Executor$TaskRunner.$anonfun$run$5(Executor.scala:647)
	at org.apache.spark.util.SparkErrorUtils.tryWithSafeFinally(SparkErrorUtils.scala:80)
	at org.apache.spark.util.SparkErrorUtils.tryWithSafeFinally$(SparkErrorUtils.scala:77)
	at org.apache.spark.util.Utils$.tryWithSafeFinally(Utils.scala:99)
	at org.apache.spark.executor.Executor$TaskRunner.run(Executor.scala:650)
	at ja

## EX2

## EX3

In [ ]:
from pyspark.sql.window import Window

# 1. Thực hiện aggregation trên stream (Tính tổng count theo window)
# Sử dụng 5-minute tumbling window và watermark 10 phút [cite: 209, 212]
windowed_counts = ratings_stream \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(window(col("timestamp"), "5 minutes"), col("movieId")) \
    .count()

# 2. Join với dữ liệu movies tĩnh để lấy title [cite: 208, 209]
trending_df = windowed_counts.join(movies_static, "movieId")

# 3. Định nghĩa hàm xử lý cho mỗi batch (Đây là nơi xử lý Rank)
def process_top_k(batch_df, batch_id):
    # Trong đây, batch_df là một STATIC DataFrame
    window_spec = Window.partitionBy("window").orderBy(col("count").desc(), col("movieId").asc())
    
    # Thực hiện xếp hạng và lọc Top 3 [cite: 209, 210, 213]
    final_batch = batch_df.withColumn("rank", dense_rank().over(window_spec)) \
        .filter(col("rank") <= 3) \
        .select("window", "movieId", "title", "count", "rank") \
        .orderBy("window", "rank")
    
    print(f"Batch ID: {batch_id} - Trending Now (Top 3 per window)")
    final_batch.show(truncate=False)

# 4. Khởi chạy stream với foreachBatch
query3 = trending_df.writeStream \
    .foreachBatch(process_top_k) \
    .trigger(processingTime='5 seconds') \
    .start()

query3.awaitTermination(20)
query3.stop()

## EX4

In [ ]:
from pyspark.sql.functions import *

# 1. Thiết lập Watermark cho cả hai luồng (Bắt buộc cho Stream-Stream Join) [cite: 220, 224]
# Cần đặt watermark để Spark biết khi nào có thể xóa các trạng thái cũ trong bộ nhớ [cite: 34, 222]
ratings_with_wm = ratings_stream.withWatermark("timestamp", "10 minutes")
tags_with_wm = tags_stream.withWatermark("timestamp", "10 minutes")

# 2. Thực hiện Stream-Stream Join trên movieId [cite: 216, 225]
# Đối với Stream-Stream join, bạn PHẢI có một điều kiện ràng buộc về thời gian 
# Ở đây chúng ta tìm các tag được gán trong khoảng +/- 5 phút so với thời điểm rating [cite: 215]
stream_stream_join = ratings_with_wm.alias("r").join(
    tags_with_wm.alias("t"),
    expr("""
        r.movieId = t.movieId AND
        t.timestamp >= r.timestamp - interval 5 minutes AND
        t.timestamp <= r.timestamp + interval 5 minutes
    """),
    "inner"
)

# 3. Chọn và định dạng các cột đầu ra theo yêu cầu [cite: 221]
# Cấu trúc: (movieId, tag, rating, ratingTime, tagTime)
result_ex4 = stream_stream_join.select(
    col("r.movieId"),
    col("t.tag"),
    col("r.rating"),
    col("r.timestamp").alias("ratingTime"),
    col("t.timestamp").alias("tagTime")
)

# 4. Ghi kết quả ra console mỗi 5 giây ở chế độ APPEND [cite: 216, 217, 221]
# Lưu ý: Stream-stream join không hỗ trợ Complete mode, chỉ dùng Append hoặc Update 
query4 = result_ex4.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='5 seconds') \
    .start()

# 5. GIỚI HẠN THỜI GIAN CHẠY 20 GIÂY
print(">>> Đang thực hiện Stream-Stream Join (Ratings & Tags) trong 20 giây...")
query4.awaitTermination(20)

# Dừng query sau khi hết 20 giây demo
query4.stop()
print(">>> Demo Exercise 4 đã dừng thành công.")